# 02 - PREPROCESSING - PlantVillage

## Mục tiêu

Tạo **một bộ dữ liệu đầu vào nhất quán** để `03_simple_cnn.ipynb`, `04_complex_cnn.ipynb` và `05_transfer_learning.ipynb` sử dụng chung.


1. Đọc `metadata.csv` và các danh sách vấn đề đã sinh ở bước EDA.
2. Loại ảnh lỗi khỏi **danh sách sử dụng** nhưng không xóa ảnh gốc.
3. Kiểm tra nhãn mâu thuẫn trong các nhóm ảnh trùng/gần trùng.
4. Loại bản sao trùng hoàn toàn khỏi danh sách sử dụng.
5. Gom ảnh trùng/gần trùng thành `group_id` để cùng một lá không rơi vào nhiều tập.
6. Giữ `val` gốc làm **test**, đồng thời chuyển cả nhóm sang test nếu nhóm đó có thành viên thuộc `val` gốc.
7. Tách phần còn lại của `train` gốc thành **87.5% train / 12.5% validation** bằng chia theo lớp và theo nhóm.
8. Cố định `SEED = 42` và lưu `data_split.csv` để mọi model dùng đúng cùng một split.
9. Xây dựng pipeline đọc ảnh theo batch, RGB, resize `224 × 224`.
10. Chỉ augmentation tập train.
11. Lưu cấu hình, class order và split.

### Tỉ lệ mục tiêu trước làm sạch

| Tập | Nguồn | Số ảnh trước làm sạch |
|---|---|---:|
| Train | 87.5% của `train` gốc | ~38,013 |
| Validation | 12.5% của `train` gốc | ~5,431 |
| Test | toàn bộ `val` gốc | 10,861 |

Do ảnh lỗi, duplicate và nhóm ảnh gần trùng có thể làm thay đổi cách chia, **số lượng thực tế sau làm sạch có thể khác bảng trên**.

C?c b??c x? l? d?ng h?m chung trong `src/data_utils.py`; notebook hi?n th? k?t qu? v? nh?n x?t.


# I. IMPORT VÀ CẤU HÌNH

In [21]:
from pathlib import Path
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32

# False = d?ng l?i data_split.csv n?u n? ?? t?n t?i v? kh?p metadata.
FORCE_REBUILD_SPLIT = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)


PyTorch version: 2.14.0+cpu
Device: cpu


# II. ĐƯỜNG DẪN PROJECT, DATASET VÀ OUTPUT

In [22]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROJECT_DIR = PROJECT_DIR.resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data_utils import (
    load_preprocessing_inputs,
    find_broken_and_missing_images,
    build_duplicate_groups,
    clean_metadata,
    build_class_mapping,
    load_or_create_split,
    validate_split,
    make_transforms,
    make_dataloaders,
    save_preprocessing_outputs,
)

load_dotenv(PROJECT_DIR / ".env")

data_dir_value = os.getenv("DATA_DIR")
DATA_DIR = Path(data_dir_value) if data_dir_value else PROJECT_DIR.parent / "PlantVillage"
if not DATA_DIR.is_absolute():
    DATA_DIR = PROJECT_DIR / DATA_DIR
DATA_DIR = DATA_DIR.resolve()

METADATA_DIR = PROJECT_DIR / "data" / "metadata"
RESULTS_DIR = PROJECT_DIR / "outputs" / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = METADATA_DIR / "metadata.csv"
DUPLICATE_PATH = METADATA_DIR / "duplicate_images.csv"
NEAR_DUPLICATE_PATH = METADATA_DIR / "near_duplicates.csv"
BROKEN_PATH = METADATA_DIR / "broken_images.csv"

DATA_SPLIT_PATH = RESULTS_DIR / "data_split.csv"
CLASS_NAMES_PATH = RESULTS_DIR / "class_names.json"
CONFIG_PATH = RESULTS_DIR / "preprocessing_config.json"

print("PROJECT_DIR :", PROJECT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("METADATA_DIR:", METADATA_DIR)
print()
print("metadata.csv exists       :", METADATA_PATH.exists())
print("duplicate_images.csv exists:", DUPLICATE_PATH.exists())
print("near_duplicates.csv exists :", NEAR_DUPLICATE_PATH.exists())

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy DATA_DIR: {DATA_DIR}")

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy data/metadata/metadata.csv. "
        "Hãy chạy 01_eda.ipynb trước."
    )

if not DUPLICATE_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy duplicate_images.csv. "
        "Hãy chạy phần kiểm tra duplicate trong 01_eda.ipynb trước."
    )

if not NEAR_DUPLICATE_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy near_duplicates.csv. "
        "Hãy chạy phần kiểm tra near duplicate trong 01_eda.ipynb trước."
    )


PROJECT_DIR : D:\Plant_disease\plant-disease-classification
DATA_DIR    : D:\Plant_disease\PlantVillage
METADATA_DIR: D:\Plant_disease\plant-disease-classification\data\metadata

metadata.csv exists       : True
duplicate_images.csv exists: True
near_duplicates.csv exists : True


# III. ĐỌC METADATA VÀ DANH SÁCH VẤN ĐỀ TỪ EDA

Các file đầu vào mong đợi:

```text
data/metadata/
├── metadata.csv
├── duplicate_images.csv
├── near_duplicates.csv
└── broken_images.csv       
```

EDA hiện tại có thể chưa lưu `broken_images.csv`. Nếu file này không tồn tại, notebook sẽ kiểm tra lại khả năng đọc ảnh để đảm bảo không đưa ảnh lỗi vào split.


In [23]:
metadata, duplicate_df, near_duplicate_df, broken_df = load_preprocessing_inputs(
    METADATA_PATH, DUPLICATE_PATH, NEAR_DUPLICATE_PATH, BROKEN_PATH
)

print("Metadata rows       :", len(metadata))
print("Exact duplicate rows:", len(duplicate_df))
print("Near-duplicate pairs:", len(near_duplicate_df))
print("Broken rows imported:", len(broken_df))
display(metadata.head())


Metadata rows       : 54305
Exact duplicate rows: 42
Near-duplicate pairs: 1165
Broken rows imported: 0


,relative_path,split,class_name,plant,condition,is_healthy,original_split
0,train/Apple___Apple_scab/01a66316-0e98-4d3b-a5...,train,Apple___Apple_scab,Apple,Apple_scab,False,train
1,train/Apple___Apple_scab/01f3deaa-6143-4b6c-9c...,train,Apple___Apple_scab,Apple,Apple_scab,False,train
2,train/Apple___Apple_scab/0208f4eb-45a4-4399-90...,train,Apple___Apple_scab,Apple,Apple_scab,False,train
3,train/Apple___Apple_scab/023123cb-7b69-4c9f-a5...,train,Apple___Apple_scab,Apple,Apple_scab,False,train
4,train/Apple___Apple_scab/0261a6e4-21f8-481a-88...,train,Apple___Apple_scab,Apple,Apple_scab,False,train


# IV. KIỂM TRA ẢNH LỖI / FILE KHÔNG TỒN TẠI

In [24]:
if not BROKEN_PATH.exists():
    print("EDA ch?a l?u broken_images.csv -> ?ang ki?m tra l?i kh? n?ng ??c ?nh...")

broken_df, broken_paths, missing_file_paths = find_broken_and_missing_images(
    metadata,
    DATA_DIR,
    broken_df=broken_df,
    verify_images=not BROKEN_PATH.exists(),
)

print("Broken images:", len(broken_paths))
print("Missing files :", len(missing_file_paths))
if len(broken_df) > 0:
    display(broken_df.head())


EDA ch?a l?u broken_images.csv -> ?ang ki?m tra l?i kh? n?ng ??c ?nh...


Broken images: 0
Missing files : 0


# V. GOM NHÓM ẢNH TRÙNG / GẦN TRÙNG VÀ KIỂM TRA NHÃN MÂU THUẪN

Ta dùng **connected components**:

- ảnh có cùng SHA256 → cùng nhóm;
- cặp near-duplicate từ EDA → cùng nhóm;
- nếu một nhóm liên thông chứa nhiều `class_name`, nhóm đó bị xem là **label conflict** và loại khỏi danh sách sử dụng;
- mọi ảnh còn lại trong cùng `group_id` phải nằm chung một split.

Cách này ngăn cùng một ảnh/cùng một lá xuất hiện ở cả train, validation và test.


In [25]:
metadata, conflict_groups, label_conflict_df = build_duplicate_groups(
    metadata, duplicate_df, near_duplicate_df
)

print("T?ng group:", metadata["group_id"].nunique())
print("Group ch?a >1 nh?n:", len(conflict_groups))
print("?nh thu?c group nh?n m?u thu?n:", len(label_conflict_df))
if len(label_conflict_df) > 0:
    display(label_conflict_df[
        ["relative_path", "class_name", "original_split", "group_id"]
    ].head(30))


T?ng group: 53318
Group ch?a >1 nh?n: 0
?nh thu?c group nh?n m?u thu?n: 0


# VI. LÀM SẠCH DANH SÁCH ẢNH

Quy tắc:

1. Loại ảnh lỗi hoặc file bị thiếu.
2. Loại toàn bộ nhóm có nhãn mâu thuẫn.
3. Với ảnh **trùng hoàn toàn** (cùng SHA256), chỉ giữ một bản trong danh sách:
   - nếu nhóm exact-duplicate có ảnh từ `val` gốc, ưu tiên giữ bản thuộc `val` để bảo toàn test;
   - nếu không, giữ bản có `relative_path` nhỏ nhất theo thứ tự từ điển.
4. Near-duplicate không bị xóa mặc định; chúng được giữ nhưng bắt buộc cùng `group_id`.

Không file ảnh gốc nào bị xóa khỏi ổ đĩa.


In [26]:
clean_df, excluded_reasons, exact_duplicate_removed = clean_metadata(
    metadata, duplicate_df, broken_paths, missing_file_paths, label_conflict_df
)

print("?nh metadata ban ??u :", len(metadata))
print("?nh l?i/file thi?u   :", len(broken_paths | missing_file_paths))
print("?nh conflict lo?i    :", len(label_conflict_df))
print("Exact copies lo?i    :", len(exact_duplicate_removed))
print("?nh c?n s? d?ng      :", len(clean_df))

reason_counts = {}
for reasons in excluded_reasons.values():
    for reason in reasons:
        reason_counts[reason] = reason_counts.get(reason, 0) + 1

print("\nChi ti?t l? do lo?i:")
for reason, count in sorted(reason_counts.items()):
    print(f"  {reason}: {count}")


?nh metadata ban ??u : 54305
?nh l?i/file thi?u   : 0
?nh conflict lo?i    : 0
Exact copies lo?i    : 21
?nh c?n s? d?ng      : 54284

Chi ti?t l? do lo?i:
  exact_duplicate_copy: 21


# VII. CHUẨN HÓA NHÃN

Mapping lớp được cố định bằng cách **sắp xếp tên class theo alphabet**.

`class_names.json` lưu đúng thứ tự này.  
Mã số class chính là vị trí của class trong danh sách, từ `0` đến `37`.


In [27]:
clean_df, class_names, class_to_idx, idx_to_class = build_class_mapping(
    metadata, clean_df
)
mapping_df = pd.DataFrame({
    "class_id": range(len(class_names)),
    "class_name": class_names,
})
print("Number of classes:", len(class_names))
display(mapping_df)


Number of classes: 38


,class_id,class_name
0,0,Apple___Apple_scab
1,1,Apple___Black_rot
2,2,Apple___Cedar_apple_rust
3,3,Apple___healthy
4,4,Blueberry___healthy
5,5,Cherry_(including_sour)___Powdery_mildew
6,6,Cherry_(including_sour)___healthy
7,7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...
8,8,Corn_(maize)___Common_rust_
9,9,Corn_(maize)___Northern_Leaf_Blight


# VIII. CHIA TRAIN / VALIDATION / TEST

## Quy tắc chia

- **Test:** toàn bộ ảnh `val` gốc.
- Nếu một `group_id` có ít nhất một ảnh từ `val` gốc, **toàn bộ group** được đưa vào test để tránh leakage.
- Phần còn lại phải đến từ `train` gốc.
- `train` gốc còn lại được chia:
  - `87.5%` train;
  - `12.5%` validation.
- Dùng `StratifiedGroupKFold(n_splits=8)`:
  - `stratify` theo `class_name`;
  - `group` theo `group_id`;
  - `shuffle=True`;
  - `random_state=42`.

Một fold trong 8 fold tương ứng xấp xỉ `12.5%`, phù hợp với validation mục tiêu.


In [28]:
split_df, selected_fold, test_group_count, use_saved_split = load_or_create_split(
    clean_df,
    DATA_SPLIT_PATH,
    force_rebuild=FORCE_REBUILD_SPLIT,
    seed=SEED,
    config_path=CONFIG_PATH,
)

print("?ang d?ng l?i split ?? l?u:" if use_saved_split else "?? t?o split m?i (s? l?u sau ki?m tra):",
      DATA_SPLIT_PATH)
print("Selected fold:", selected_fold)
print("Test groups  :", test_group_count)
display(split_df["split"].value_counts().rename("count").to_frame())


?ang d?ng l?i split ?? l?u: D:\Plant_disease\plant-disease-classification\outputs\results\data_split.csv
Selected fold: 4
Test groups  : 10763


,count
split,
train,37538
test,11383
validation,5363


# IX. KIỂM TRA BỘ CHIA

In [29]:
validation_report = validate_split(
    split_df, near_duplicate_df, class_names
)
leaking_groups = validation_report["leaking_groups"]
near_pair_leaks = validation_report["near_pair_leaks"]
split_summary = validation_report["split_summary"]
class_split_counts = validation_report["class_split_counts"]

print("Kh?ng ph?t hi?n path/group overlap gi?a c?c split.")
print()
display(split_summary)
display(class_split_counts)


Kh?ng ph?t hi?n path/group overlap gi?a c?c split.



,count,ratio
split,,
train,37538,0.6915
validation,5363,0.0988
test,11383,0.2097


,split,train,validation,test,total
class_id,class_name,,,,
0,Apple___Apple_scab,441,63,126,630
1,Apple___Black_rot,434,62,125,621
2,Apple___Cedar_apple_rust,193,27,55,275
3,Apple___healthy,1145,163,330,1638
4,Blueberry___healthy,1050,150,302,1502
5,Cherry_(including_sour)___Powdery_mildew,734,105,213,1052
6,Cherry_(including_sour)___healthy,593,85,176,854
7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,359,51,103,513
8,Corn_(maize)___Common_rust_,834,119,239,1192


In [30]:
missing_class_rows = validation_report["missing_class_rows"]
if missing_class_rows:
    print("C?NH B?O: c? class thi?u trong m?t split:")
    for item in missing_class_rows:
        print(item["split"], "->", item["missing_classes"])
else:
    print("OK: c? train, validation v? test ??u c? ??", len(class_names), "class.")


OK: c? train, validation v? test ??u c? ?? 38 class.


# X. PIPELINE ĐỌC ẢNH

`data_split.csv` là nguồn duy nhất quyết định ảnh thuộc train, validation hay test.

Pipeline preprocessing:

- đọc ảnh theo yêu cầu, không đưa toàn bộ dataset vào RAM;
- chuyển ảnh sang `RGB`;
- resize về `224 × 224`;
- train: lật ngang, xoay, zoom nhẹ và điều chỉnh tương phản nhẹ;
- chuyển ảnh thành tensor với giá trị pixel trong `[0, 1]`;
- validation/test: chỉ resize + chuyển tensor, không augmentation.


In [31]:
train_transform, eval_transform = make_transforms(image_size=IMG_SIZE)
print("Train transform:")
print(train_transform)
print("\nValidation/Test transform:")
print(eval_transform)


Train transform:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    RandomAffine(degrees=[0.0, 0.0], scale=(0.9, 1.1))
    ColorJitter(brightness=None, contrast=(0.9, 1.1), saturation=None, hue=None)
    ToTensor()
)

Validation/Test transform:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
)


In [32]:
train_split_df = split_df[split_df["split"] == "train"].copy()
validation_split_df = split_df[split_df["split"] == "validation"].copy()
test_split_df = split_df[split_df["split"] == "test"].copy()

print("Train dataset     :", len(train_split_df))
print("Validation dataset:", len(validation_split_df))
print("Test dataset      :", len(test_split_df))


Train dataset     : 37538
Validation dataset: 5363
Test dataset      : 11383


# XI. DATALOADER / BATCH
batch giúp vừa đủ bộ nhớ để chạy, vừa cho model học dần qua nhiều bước.

In [33]:
# num_workers=0 ?n ??nh trong Jupyter/VS Code tr?n Windows.
train_loader, validation_loader, test_loader = make_dataloaders(
    split_df,
    DATA_DIR,
    train_transform,
    eval_transform,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

train_dataset = train_loader.dataset
validation_dataset = validation_loader.dataset
test_dataset = test_loader.dataset

print("Train batches     :", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches      :", len(test_loader))


Train batches     : 1174
Validation batches: 168
Test batches      : 356


# XII. KIỂM TRA MỘT BATCH SAU XỬ LÝ

In [34]:
images, labels = next(
    iter(train_loader)
)

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype      :", images.dtype)
print("Label dtype      :", labels.dtype)
print(
    "Pixel range      :",
    float(images.min()),
    "->",
    float(images.max())
)
print(
    "Label range      :",
    int(labels.min()),
    "->",
    int(labels.max())
)

assert images.ndim == 4
assert images.shape[1] == 3
assert images.shape[2] == IMG_SIZE
assert images.shape[3] == IMG_SIZE
assert float(images.min()) >= 0.0
assert float(images.max()) <= 1.0


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Image dtype      : torch.float32
Label dtype      : torch.int64
Pixel range      : 0.0 -> 0.9686274528503418
Label range      : 1 -> 36


# XIII. LƯU CLASS ORDER VÀ CẤU HÌNH

In [35]:
split_count_dict = {
    name: int(
        (split_df["split"] == name).sum()
    )
    for name in [
        "train",
        "validation",
        "test"
    ]
}

config = {
    "seed": SEED,
    "image_size": [
        IMG_SIZE,
        IMG_SIZE
    ],
    "batch_size": BATCH_SIZE,
    "num_classes": len(class_names),
    "paths": {
        "metadata": str(
            METADATA_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "duplicate_issues": str(
            DUPLICATE_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "near_duplicate_issues": str(
            NEAR_DUPLICATE_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "data_split": str(
            DATA_SPLIT_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "class_names": str(
            CLASS_NAMES_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/")
    },
    "cleaning": {
        "metadata_rows_before_cleaning": int(
            len(metadata)
        ),
        "broken_images": int(
            len(broken_paths)
        ),
        "missing_files": int(
            len(missing_file_paths)
        ),
        "label_conflict_groups": int(
            len(conflict_groups)
        ),
        "label_conflict_images": int(
            len(label_conflict_df)
        ),
        "exact_duplicate_copies_removed": int(
            len(exact_duplicate_removed)
        ),
        "rows_after_cleaning": int(
            len(clean_df)
        )
    },
    "split_strategy": {
        "test": (
            "All usable original val images; "
            "if a duplicate/near-duplicate group touches original val, "
            "the whole group is assigned to test."
        ),
        "train_validation_source": (
            "Usable original train images not assigned to a test group."
        ),
        "validation_fraction_of_remaining_train": 0.125,
        "splitter": "StratifiedGroupKFold",
        "n_splits": 8,
        "selected_fold": (
            None
            if selected_fold is None
            else int(selected_fold)
        ),
        "stratify_column": "class_name",
        "group_column": "group_id"
    },
    "final_split_counts": split_count_dict,
    "custom_cnn_preprocessing": {
        "color": "RGB",
        "resize": [
            IMG_SIZE,
            IMG_SIZE
        ],
        "pixel_range": [
            0.0,
            1.0
        ],
        "train_augmentation": {
            "horizontal_flip_probability": 0.5,
            "rotation_degrees": 10,
            "zoom_scale": [
                0.90,
                1.10
            ],
            "contrast_jitter": 0.10
        },
        "validation_augmentation": None,
        "test_augmentation": None
    },
    "transfer_learning_preprocessing": (
        "Use the preprocessing/normalization required by the selected "
        "pretrained weights in the transfer-learning notebook."
    )
}

save_preprocessing_outputs(
    split_df,
    class_names,
    config,
    DATA_SPLIT_PATH,
    CLASS_NAMES_PATH,
    CONFIG_PATH,
    save_split=not use_saved_split,
)

print("Reused split:" if use_saved_split else "Saved split:", DATA_SPLIT_PATH)
print("Saved:", CLASS_NAMES_PATH)
print("Saved:", CONFIG_PATH)


Reused split: D:\Plant_disease\plant-disease-classification\outputs\results\data_split.csv
Saved: D:\Plant_disease\plant-disease-classification\outputs\results\class_names.json
Saved: D:\Plant_disease\plant-disease-classification\outputs\results\preprocessing_config.json


# XIV. TỔNG KẾT

In [36]:
print("=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)

print(
    f"Metadata trước làm sạch : {len(metadata):,}"
)
print(
    f"Dữ liệu sau làm sạch    : {len(split_df):,}"
)
print(
    f"Number of classes       : {len(class_names)}"
)

print()

for split_name in [
    "train",
    "validation",
    "test"
]:
    count = int(
        (split_df["split"] == split_name).sum()
    )

    ratio = (
        count / len(split_df)
        if len(split_df) > 0
        else 0
    )

    print(
        f"{split_name:<11}: "
        f"{count:>6,} "
        f"({ratio:.2%})"
    )

print()
print(
    "Group leakage            : 0"
)
print(
    "Near-duplicate leakage   : 0"
)
print(
    "Image size               : "
    f"{IMG_SIZE} x {IMG_SIZE}"
)
print(
    "Custom CNN pixel range   : [0, 1]"
)
print(
    "Train augmentation       : Yes"
)
print(
    "Validation/Test augment  : No"
)

print("\nOutputs:")
print(" -", DATA_SPLIT_PATH)
print(" -", CLASS_NAMES_PATH)
print(" -", CONFIG_PATH)


PREPROCESSING SUMMARY
Metadata trước làm sạch : 54,305
Dữ liệu sau làm sạch    : 54,284
Number of classes       : 38

train      : 37,538 (69.15%)
validation :  5,363 (9.88%)
test       : 11,383 (20.97%)

Group leakage            : 0
Near-duplicate leakage   : 0
Image size               : 224 x 224
Custom CNN pixel range   : [0, 1]
Train augmentation       : Yes
Validation/Test augment  : No

Outputs:
 - D:\Plant_disease\plant-disease-classification\outputs\results\data_split.csv
 - D:\Plant_disease\plant-disease-classification\outputs\results\class_names.json
 - D:\Plant_disease\plant-disease-classification\outputs\results\preprocessing_config.json


## Output để các notebook model đọc độc lập

Các notebook model **không cần chạy lại notebook này** nếu các file output đã tồn tại.

### `outputs/results/data_split.csv`

Các cột chính:

```text
relative_path
class_name
class_id
split
original_split
group_id
```

### `outputs/results/class_names.json`

Danh sách tên class theo đúng thứ tự `class_id`.

### `outputs/results/preprocessing_config.json`

Lưu seed, kích thước ảnh, batch size, cleaning summary, split strategy và augmentation.

Ba model sau đó phải đọc lại `data_split.csv` và `class_names.json` thay vì tự chia dữ liệu lần nữa.
